# Домашняя работа. VAE + NF + VAPNEV

Colab-версия задания HSE-LAMBDA. Сохранены задания, разрешение **64×64** и требования к сдаче. Шкала переведена в **10 баллов**.

| Задание | Баллы |
|---|---:|
| Нормализующий поток | 3 |
| VAE | 2 |
| VAPNEV | 3 |
| Conditional VAPNEV | 1 |
| Анализ | 1 |
| **Всего** | **10** |

1. Сохраните копию ноутбука в Google Drive. Выберите GPU в настройках среды выполнения.
2. Загрузите выданный преподавателем `celeba.zip` в Colab либо укажите путь к нему на Google Drive.
3. Укажите путь к отдельному архиву атрибутов или к `list_attr_celeba.txt`, если он выдан отдельно.
4. Выполните ячейки подготовки и визуализации данных. Далее заполните исходные `TODO` и проведите эксперименты.

Для большого ZIP удобнее использовать файл на Drive. При загрузке через левую панель «Файлы» Colab архив обычно доступен как `/content/celeba.zip`. Данные в `/content` хранятся до завершения сессии.

Загрузчик `CelebADataset` из курса включён в ноутбук с адаптацией путей и загрузки локальных архивов. Подготовка работает без клонирования репозитория.

Исходник — [HSE-LAMBDA, spring-2025-long](https://github.com/HSE-LAMBDA/DeepGenerativeModels/blob/42dd64b535f5bedf1d2b3db28f2f21ce96c9f09f/homeworks/03_hw_vapnev.ipynb). Атрибуция и лицензия приведены в конце.

# Домашнее задание 3. VAE + NF + VAPNEV

В этом задании вам предстоит поработать с набором данных CelebA. Решение домашней работы должно быть представлено в виде .zip архива, содержащего:
- Jupyter Notebook(и) и/или модули с кодом,
- PDF-отчет, в котором необходимо описать проведённые эксперименты, а также обосновать выбор используемых методов (например, дополнительные лоссы для стабилизации, виды потоков, особенности реализации VAE и пр.).

## Настройка окружения

In [ ]:
import importlib.util
import subprocess
import sys

packages = {"torch": "torch", "PIL": "pillow", "numpy": "numpy", "matplotlib": "matplotlib"}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

from pathlib import Path
import random
import torch
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
cuda = torch.cuda.is_available()
device = torch.device("cuda" if cuda else ("mps" if torch.backends.mps.is_available() else "cpu"))
torch.backends.cudnn.benchmark = cuda
print("Python:", sys.version.split()[0], "PyTorch:", torch.__version__)
print("Using device:", device)

In [ ]:
"""Colab adaptation of HSE-LAMBDA's CelebADataset.

Source: DeepGenerativeModels/seminars/utils/datasets/celeba.py
Revision: 42dd64b535f5bedf1d2b3db28f2f21ce96c9f09f
Copyright (c) 2024 Laboratory of methods for Big Data Analysis at HSE.
MIT License; see assets/HSE_LAMBDA_LICENSE.txt.

The original dataset interface is preserved. Data is supplied explicitly by the
instructor instead of downloaded from the upstream Google Drive links.
"""

from pathlib import Path, PurePosixPath
import shutil
import stat
import tempfile
import zipfile

from PIL import Image
import torch
from torch.utils.data import Dataset


def extract_celeba_zip(archive_path, destination):
    """Extract provided files atomically and reject paths outside destination."""
    destination = Path(destination).expanduser().resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(Path(archive_path).expanduser()) as archive:
        for member in archive.infolist():
            relative = PurePosixPath(member.filename.replace("\\", "/"))
            if relative.is_absolute() or ".." in relative.parts:
                raise ValueError(f"Invalid archive path: {member.filename}")
            if any(part.startswith(".") or part == "__MACOSX" for part in relative.parts):
                continue
            target = destination.joinpath(*relative.parts).resolve()
            if not target.is_relative_to(destination) or stat.S_ISLNK(member.external_attr >> 16):
                raise ValueError(f"Invalid archive member: {member.filename}")
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with tempfile.NamedTemporaryFile(dir=target.parent, delete=False) as output:
                temporary = Path(output.name)
                try:
                    with archive.open(member) as source:
                        shutil.copyfileobj(source, output)
                except BaseException:
                    temporary.unlink(missing_ok=True)
                    raise
            temporary.replace(target)


def find_celeba_files(search_root, attributes_path=None):
    """Locate the upstream image directory and its TXT attributes."""
    search_root = Path(search_root).expanduser()
    image_directories = sorted(
        folder for folder in search_root.rglob("img_align_celeba")
        if folder.is_dir() and any(folder.glob("*.jpg"))
    )
    if search_root.name == "img_align_celeba" and any(search_root.glob("*.jpg")):
        image_directories.insert(0, search_root)
    if len(image_directories) != 1:
        raise FileNotFoundError(
            f"Expected one img_align_celeba directory with JPG images under {search_root}; "
            f"found {len(image_directories)}. Set DATA_DIR to the extracted CelebA directory."
        )
    if attributes_path:
        attribute_file = Path(attributes_path).expanduser()
    else:
        candidates = sorted(search_root.rglob("list_attr_celeba.txt"))
        if len(candidates) != 1:
            raise FileNotFoundError(
                f"Found {len(candidates)} list_attr_celeba.txt files. "
                "Extract the attributes ZIP or specify ATTR_PATH."
            )
        attribute_file = candidates[0]
    if not attribute_file.is_file():
        raise FileNotFoundError(f"Attribute file not found: {attribute_file}")
    return image_directories[0].parent, attribute_file


class CelebADataset(Dataset):
    """HSE-LAMBDA dataset interface with explicit local images and attributes."""

    def __init__(self, root_dir="../data/celeba", transform=None, attributes_path=None):
        self.root_dir = Path(root_dir).expanduser()
        self.transform = transform
        self.dataset_folder = self.root_dir / "img_align_celeba"
        self.filenames = sorted(
            path.name for path in self.dataset_folder.glob("*")
            if path.is_file() and path.suffix.lower() in {".jpg", ".jpeg", ".png"}
        )
        if not self.filenames:
            raise FileNotFoundError(f"No images found in {self.dataset_folder}")
        attribute_file = (
            Path(attributes_path).expanduser() if attributes_path
            else self.root_dir / "annotations" / "list_attr_celeba.txt"
        )
        self.annotations = []
        with attribute_file.open(encoding="utf-8-sig") as stream:
            expected_count = int(next(stream).strip())
            self.header = next(stream).split()
            for line in stream:
                parts = line.split()
                if not parts:
                    continue
                values = [int(value) for value in parts[1:]]
                if len(values) != len(self.header) or not set(values).issubset({-1, 1}):
                    raise ValueError(f"Invalid attributes for {parts[0]}")
                self.annotations.append((parts[0], values))
        if len(self.annotations) != expected_count:
            raise ValueError("Attribute row count does not match the file header.")
        self.attr_map = {
            filename: torch.tensor(values, dtype=torch.long)
            for filename, values in self.annotations
        }
        if len(self.attr_map) != len(self.annotations):
            raise ValueError("Duplicate filenames in the attributes file.")
        missing = [name for name in self.filenames if name not in self.attr_map]
        if missing:
            raise ValueError(f"Missing attributes for {len(missing)} images, including {missing[:3]}")

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        filename = self.filenames[idx]
        with Image.open(self.dataset_folder / filename) as source:
            image = source.convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, {
            "filename": filename,
            "idx": idx,
            "attributes": self.attr_map[filename],
        }

## Архив изображений и атрибуты

Исходный загрузчик курса ожидает следующую структуру

```text
celeba/
├── img_align_celeba/
│   ├── 000001.jpg
│   └── ...
└── annotations/
    └── list_attr_celeba.txt
```

`IMAGE_ZIP` — путь к выданному архиву. `ATTR_ZIP` — путь к отдельному ZIP с атрибутами, если он есть. `ATTR_PATH` — путь к уже распакованному `list_attr_celeba.txt`.

`DATA_DIR` задаёт каталог распаковки. Дополнительная папка `celeba` внутри архива поддерживается. Если данные уже распакованы, оставьте оба пути ZIP пустыми и укажите их каталог в `DATA_DIR`. При нескольких наборах данных сузьте каталог поиска, при нескольких файлах атрибутов задайте `ATTR_PATH` явно.

Для Drive включите `USE_GOOGLE_DRIVE` и укажите путь вида `/content/drive/MyDrive/celeba.zip`. Файлы распаковываются в локальный `DATA_DIR`, чтобы чтение batch не зависело от скорости Drive. При повторном запуске подготовки можно выключить `EXTRACT_ARCHIVES`.

In [ ]:
#@title Dataset paths
USE_GOOGLE_DRIVE = False #@param {type:"boolean"}
IMAGE_ZIP = "/content/celeba.zip" #@param {type:"string"}
ATTR_ZIP = "" #@param {type:"string"}
ATTR_PATH = "" #@param {type:"string"}
DATA_DIR = "/content/DeepGenerativeModels/seminars/data/celeba" #@param {type:"string"}
EXTRACT_ARCHIVES = True #@param {type:"boolean"}

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

if EXTRACT_ARCHIVES:
    for archive_path in (IMAGE_ZIP, ATTR_ZIP):
        if archive_path.strip():
            if not Path(archive_path).expanduser().is_file():
                raise FileNotFoundError(f"Upload the archive and update its path: {archive_path}")
            print("Extracting:", archive_path)
            extract_celeba_zip(archive_path, DATA_DIR)
    # The upstream dataset may also be supplied as an inner image ZIP.
    for image_archive in Path(DATA_DIR).expanduser().rglob("img_align_celeba.zip"):
        extract_celeba_zip(image_archive, image_archive.parent)

dataset_root, attribute_file = find_celeba_files(DATA_DIR, ATTR_PATH or None)
print("Dataset root:", dataset_root)
print("Attributes:", attribute_file)

## Загрузка данных

In [ ]:
# Resize the shorter side to 64, center-crop, then normalize to [-1, 1].
# Equivalent preprocessing stages to the original torchvision pipeline.
t_normalize = lambda x: x * 2 - 1
t_invnormalize = lambda x: (x + 1) / 2

def transform(image):
    image = image.convert("RGB")
    width, height = image.size
    if width <= height:
        resized = (64, int(64 * height / width))
    else:
        resized = (int(64 * width / height), 64)
    image = image.resize(resized, resample=Image.Resampling.BILINEAR)
    left = int(round((image.width - 64) / 2))
    top = int(round((image.height - 64) / 2))
    image = image.crop((left, top, left + 64, top + 64))
    pixels = np.array(image, dtype=np.float32) / 255.0
    tensor = torch.from_numpy(pixels).permute(2, 0, 1).contiguous()
    return t_normalize(tensor)

In [ ]:
# Preserve the original image-only interface for unconditional models.
class CelebaCustomDataset(CelebADataset):
    def __getitem__(self, idx):
        image, _ = super().__getitem__(idx)
        return image

dataset = CelebaCustomDataset(
    transform=transform,
    root_dir=dataset_root,
    attributes_path=attribute_file,
)
img = dataset[0]
assert img.shape == (3, 64, 64)
assert torch.isfinite(img).all() and img.min() >= -1 and img.max() <= 1
print("Images:", len(dataset), "Image shape:", tuple(img.shape))
print("Attribute names:", dataset.header)

`CelebaCustomDataset` возвращает только изображение, как в исходном задании. Для conditional VAPNEV используйте `CelebADataset` с теми же аргументами. Он возвращает `(image, info)`. Вектор `info["attributes"]` хранит значения −1/+1 в порядке `dataset.header`. Для бинарного условия можно преобразовать выбранное значение по формуле `(value + 1) // 2`.

### Визуализация датасета

In [ ]:
def plot_img(img, transform):
    plt.imshow(transform(img).permute(1, 2, 0))
    plt.axis('off')

In [ ]:
plt.figure(figsize=(15, 15))
for i in range(25):
    random_idx = np.random.randint(len(dataset))
    img = dataset[random_idx]
    plt.subplot(5, 5, i + 1)
    plot_img(img, t_invnormalize)
plt.show()

### Задача 1 (3/10 баллов): Построение и обучение нормального потока (Normalizing Flow) на CelebA

Необходимо обучить нормальный поток до качества, удовлетворяющего базовым порогам (точные пороги будут предоставлены позже). 

Измерьте FID и Negative Log Likelihood для оценки модели.

Внутри потока можно использовать любые компоненты: Coupling/Autoregressive/Linear слои, произвольные трансформации и т.д. Главное — реализовать их самостоятельно без копипаста.

Можно использовать как сверточные потоки (обычно дают лучшие результаты, но сложнее реализуются), так и линейные потоки (будут менее качественными), при этом можно предварительно свести изображение к вектору.

Можно использовать готовый код из репозитория курса, но рекомендуется написать свою реализацию.

In [ ]:
# TODO: Реализуйте нормальный поток для CelebA.

### Задача 2 (2/10 баллов): Построение и обучение VAE на CelebA

Обучите вариационный автоэнкодер (VAE) до достижения приемлемого качества генерации. Измерьте FID и сохраните результаты для дальнейшего сравнения.

Можно использовать готовые решения из репозитория курса, но предпочтительно написать свою реализацию.

In [ ]:
# TODO: Реализуйте VAE для CelebA.

### Задача 3 (4/10 баллов)

#### Задача 3.1 (3 балла): Построение и обучение VAPNEV на CelebA

Изучите статью [VAPNEV](https://arxiv.org/pdf/1611.05209.pdf), реализуйте и обучите модель до качества, удовлетворяющего базовым порогам (точные пороги будут предоставлены позже). Измерьте FID и сохраните результаты.

#### Задача 3.2 (1 балл): Построение и обучение conditional VAPNEV на CelebA

Реализуйте условный (conditional) вариант VAPNEV. Подробности можно найти в описании задачи и в статье.

In [ ]:
# TODO: Реализуйте VAPNEV и conditional VAPNEV для CelebA.

### Задача 4 (1/10 баллов): Анализ

Проведите анализ латентного пространства и оцените качество генерации:
1. Определите, у какой модели лучше интерполируется латентное пространство.
2. Попробуйте взять внешнее изображение (не из CelebA), отобразить его в латентное пространство и найдите ближайшие по значению векторы.
3. Продемонстрируйте самый интересный (или забавный) результат генерации, который у вас получался.

In [ ]:
# TODO: Добавьте код для анализа латентного пространства и демонстрации результатов.

### Задача 0: Отчет

Для получения полного балла по каждому пункту задания необходимо включить в отчет (предоставляемый отдельно в формате PDF) примеры генераций моделей. 

Сравните результаты различных моделей, постарайтесь объяснить, почему одни подходы работают лучше других.

## Источники и атрибуция

- [Исходная домашняя работа](https://github.com/HSE-LAMBDA/DeepGenerativeModels/blob/42dd64b535f5bedf1d2b3db28f2f21ce96c9f09f/homeworks/03_hw_vapnev.ipynb).
- [Исходный загрузчик CelebADataset](https://github.com/HSE-LAMBDA/DeepGenerativeModels/blob/42dd64b535f5bedf1d2b3db28f2f21ce96c9f09f/seminars/utils/datasets/celeba.py). Адаптированы пути и чтение выданных архивов, сохранён интерфейс данных и атрибутов.
- [CelebA](https://mmlab.ie.cuhk.edu.hk/projects/CelebA.html).
- [Подробная теория VAE](https://fpakhurov.com/ru/notes/generative-models/vae/) · [нормализующие потоки](https://fpakhurov.com/ru/notes/generative-models/normalizing-flows/) · [архитектуры потоков](https://fpakhurov.com/ru/notes/generative-models/flow-architectures/).

Исходные материалы HSE-LAMBDA распространяются по MIT License.

MIT License

Copyright (c) 2024 Laboratory of methods for Big Data Analysis at HSE

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.